# Notebook 11 — RAG Basics

Chunk a document, build a simple vector store, and retrieve relevant context before answering.

In [ ]:
import sys, os, math, re
from collections import Counter
sys.path.insert(0, os.path.abspath(".."))
from utils.anthropic_client import client, FAST_MODEL


## Step 1 — Chunking

In [ ]:
DOCUMENT = """
Python was created by Guido van Rossum and first released in 1991.
It emphasises code readability using indentation.

Python 3.0, released in 2008, was a backwards-incompatible major revision.
Python 2 reached end-of-life in April 2020.

Python supports multiple paradigms: OOP, functional, imperative.
It has a large standard library called "batteries included".
"""

def chunk_paragraphs(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

chunks = chunk_paragraphs(DOCUMENT)
print(f"Number of chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:60]}...")


## Step 2 — Simple TF-IDF embedding

In [ ]:
def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

def build_tfidf(chunks):
    vocab = sorted({w for c in chunks for w in tokenize(c)})
    n = len(chunks)
    idf = {}
    for w in vocab:
        df = sum(1 for c in chunks if w in tokenize(c))
        idf[w] = math.log((n+1)/(df+1)) + 1
    def embed(text):
        tf = Counter(tokenize(text))
        total = len(tokenize(text)) or 1
        return [tf.get(w,0)/total * idf[w] for w in vocab]
    return embed

embed = build_tfidf(chunks)
chunk_vecs = [embed(c) for c in chunks]
print("Vector dimension:", len(chunk_vecs[0]))


## Step 3 — Cosine similarity search

In [ ]:
def cosine(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    mag = lambda v: math.sqrt(sum(x*x for x in v))
    return dot/(mag(a)*mag(b)) if mag(a) and mag(b) else 0

def search(query, top_k=2):
    q_vec = embed(query)
    scored = sorted(enumerate(chunk_vecs), key=lambda x: cosine(q_vec, x[1]), reverse=True)
    return [(chunks[i], cosine(q_vec, chunk_vecs[i])) for i,_ in scored[:top_k]]

hits = search("Who created Python?")
for chunk, score in hits:
    print(f"[{score:.3f}] {chunk[:80]}")


## Step 4 — RAG answer

In [ ]:
def rag_answer(question):
    context = "\n\n".join(c for c, _ in search(question))
    r = client.messages.create(
        model=FAST_MODEL, max_tokens=200,
        messages=[{"role": "user",
                   "content": f"Answer ONLY from context below.\n\nContext:\n{context}\n\nQ: {question}"}]
    )
    return r.content[0].text

print(rag_answer("Who created Python?"))
print(rag_answer("What happened to Python 2?"))


## Exercise
Add your own document and test 3 questions, including one that is out-of-scope.

In [ ]:
# Your code here
